In [0]:
%sql
--DROP schema  IF EXISTS workspace.bookstore_eng_pro CASCADE;

In [0]:
exec(open('./Copy-Datasets.py').read())

In [0]:
#chekcing dataset_bookstore


dataset_bookstore ='/Volumes/workspace/bookstore_eng_pro/dataset'
files = dbutils.fs.ls(f"{dataset_bookstore}/kafka-raw")
display(files)

In [0]:
df_raw = spark.read.json(f"{dataset_bookstore}/kafka-raw")
display(df_raw)

In [0]:
from pyspark.sql import functions as F

def process_bronze():
  
    schema = "key BINARY, value BINARY, topic STRING, partition LONG, offset LONG, timestamp LONG"

    query = (spark.readStream
                        .format("cloudFiles")
                        .option("cloudFiles.format", "json")
                        .schema(schema)
                        .load(f"{bookstore.dataset_path}/kafka-raw")
                        .withColumn("timestamp", (F.col("timestamp")/1000).cast("timestamp"))  
                        .withColumn("year_month", F.date_format("timestamp", "yyyy-MM"))
                  .writeStream
                      .option("checkpointLocation", f"{bookstore.checkpoint_path}/bronze")
                      .option("mergeSchema", True)
                      .partitionBy("topic", "year_month")
                      .trigger(availableNow=True)
                      .table("bronze"))
    
    query.awaitTermination()

In [0]:
bookstore.process_bronze()

In [0]:
batch_df = spark.table("bronze")
display(batch_df)

In [0]:
%sql
select distinct topic from bronze

In [0]:
bookstore.load_new_data()

In [0]:
bookstore.process_bronze()

In [0]:
%sql
select count(*) from bronze

In [0]:
%sql
SELECT cast(key AS STRING), cast(value AS STRING)
FROM bronze
LIMIT 20


# Databricks notebook source

<div  style="text-align: center; line-height: 0; padding-top: 9px;">
<img src="https://raw.githubusercontent.com/derar-alhussein/Databricks-Certified-Data-Engineer-Professional/main/Includes/images/orders.png" width="60%">
</div>

In [0]:
 %sql
SELECT v.*
FROM (
SELECT from_json(cast(value AS STRING), "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>") v
FROM bronze
WHERE topic = "orders")

In [0]:
(spark.readStream
      .table("bronze")
      .createOrReplaceTempView("bronze_tmp"))


In [0]:
%sql
SELECT v.*
    FROM (
    SELECT from_json(cast(value AS STRING), "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>") v
    FROM bronze_tmp
    WHERE topic = "orders")


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW orders_silver_tmp AS
   SELECT v.*
 FROM (
SELECT from_json(cast(value AS STRING), "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>") v
FROM bronze_tmp
WHERE topic = "orders")

In [0]:
orders_silver_df = spark.sql("""
    SELECT v.*
    FROM (
    SELECT from_json(cast(value AS STRING), "order_id STRING, order_timestamp Timestamp, customer_id STRING, quantity BIGINT, total BIGINT, books ARRAY<STRUCT<book_id STRING, quantity BIGINT, subtotal BIGINT>>") v
    FROM bronze_tmp
    WHERE topic = "orders")
""")

display(orders_silver_df, checkpointLocation = f"{bookstore.checkpoint_path}/tmp/orders_silver_{time.time()}")

In [0]:
query = (spark.table("orders_silver_tmp")
               .writeStream
               .option("checkpointLocation", f"{bookstore.checkpoint_path}/orders_silver")
               .trigger(availableNow=True)
               .table("orders_silver"))

query.awaitTermination()

In [0]:
%sql
SELECT *
FROM orders_silver